In [2]:
!pip install yfinance pandas numpy matplotlib seaborn plotly \
             ta requests anthropic openpyxl \
             google-auth gspread --quiet

print("✅ All libraries installed successfully!")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 14.5 MB/s eta 0:00:00
✅ All libraries installed successfully!


In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style for charts
plt.style.use('dark_background')
sns.set_palette("husl")

# ---- FETCH M&M STOCK DATA ----
ticker = "M&M.NS"  # NSE ticker for Mahindra & Mahindra
end_date   = datetime.today()
start_date = end_date - timedelta(days=365)

print(f"📥 Fetching data from {start_date.date()} to {end_date.date()}...")

mm_stock = yf.Ticker(ticker)
df = mm_stock.history(
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    interval="1d"
)

# Reset index and clean column names
df.reset_index(inplace=True)
df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
df.columns = [col.strip() for col in df.columns]

print(f"✅ Fetched {len(df)} trading days of data")
print(f"📊 Columns: {list(df.columns)}")
df.tail(5)

📥 Fetching data from 2025-04-22 to 2026-04-22...
✅ Fetched 248 trading days of data
📊 Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
243,2026-04-15,3250.0,3301.899902,3225.000000,3256.500000,2591177,0.0,0.0
244,2026-04-16,3273.0,3288.000000,3201.000000,3222.300049,2678634,0.0,0.0
245,2026-04-17,3210.0,3242.500000,3185.000000,3200.199951,2180404,0.0,0.0
246,2026-04-20,3207.0,3235.500000,3162.600098,3221.600098,3201466,0.0,0.0
247,2026-04-21,3212.0,3262.300049,3212.000000,3247.300049,1972188,0.0,0.0


In [4]:
info = mm_stock.info

print("📋 COMPANY PROFILE")
print("="*50)
print(f"Company     : {info.get('longName', 'N/A')}")
print(f"Sector      : {info.get('sector', 'N/A')}")
print(f"Industry    : {info.get('industry', 'N/A')}")
print(f"Market Cap  : ₹{info.get('marketCap', 0)/1e9:.2f} Billion")
print(f"52W High    : ₹{info.get('fiftyTwoWeekHigh', 0):.2f}")
print(f"52W Low     : ₹{info.get('fiftyTwoWeekLow', 0):.2f}")
print(f"P/E Ratio   : {info.get('trailingPE', 'N/A')}")
print(f"EPS (TTM)   : ₹{info.get('trailingEps', 'N/A')}")
print(f"Dividend    : {info.get('dividendYield', 0)*100:.2f}%")

📋 COMPANY PROFILE
Company     : Mahindra & Mahindra Limited
Sector      : Consumer Cyclical
Industry    : Auto Manufacturers
Market Cap  : ₹3848.32 Billion
52W High    : ₹3839.90
52W Low     : ₹2826.00
P/E Ratio   : 22.952803
EPS (TTM)   : ₹139.63
Dividend    : 78.00%


In [5]:
print("🔍 Data Quality Check BEFORE Cleaning:")
print(df.isnull().sum())
print(f"Total Rows : {len(df)}")

# Drop rows with any missing values
df.dropna(inplace=True)

# Add time-based columns for Power BI
df['Year']    = df['Date'].dt.year
df['Month']   = df['Date'].dt.month_name()
df['Quarter'] = df['Date'].dt.quarter.apply(lambda x: f"Q{x}")
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Week']    = df['Date'].dt.isocalendar().week.astype(int)

# Add daily return
df['Daily_Return']    = df['Close'].pct_change() * 100
df['Cumulative_Return'] = (1 + df['Daily_Return']/100).cumprod() * 100

# Price change column
df['Price_Change'] = df['Close'] - df['Open']
df['Up_Down'] = np.where(df['Close'] > df['Open'], 'Up', 'Down')

print("✅ Data cleaned. Final shape:", df.shape)
df.describe().round(2)

🔍 Data Quality Check BEFORE Cleaning:
Date            0
Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64
Total Rows : 248
✅ Data cleaned. Final shape: (248, 17)


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Year,Week,Daily_Return,Cumulative_Return,Price_Change
count,248,248.00,248.00,248.00,248.00,248.00,248.00,248.0,248.00,248.00,247.00,247.00,248.00
mean,2025-10-17 22:44:30.967741952,3358.21,3395.13,3322.00,3357.58,2523424.99,0.10,0.0,2025.29,26.49,0.08,120.22,-0.64
min,2025-04-22 00:00:00,2742.96,2799.51,2734.03,2794.65,0.00,0.00,0.0,2025.00,1.00,-5.26,101.67,-113.50
25%,2025-07-17 18:00:00,3134.71,3171.75,3095.90,3137.20,1703397.75,0.00,0.0,2025.00,13.75,-0.79,112.43,-30.22
50%,2025-10-16 12:00:00,3379.50,3424.50,3336.70,3388.95,2237053.00,0.00,0.0,2025.00,26.50,-0.02,121.43,-0.05
75%,2026-01-15 06:00:00,3608.65,3640.40,3578.42,3605.30,3023110.00,0.00,0.0,2026.00,39.00,1.02,129.04,27.55
max,2026-04-21 00:00:00,3805.90,3839.90,3785.00,3802.40,8315834.00,25.30,0.0,2026.00,52.00,6.77,136.06,148.80
std,NaN,267.16,262.12,266.15,262.28,1246543.22,1.61,0.0,0.46,15.01,1.71,9.32,46.73


In [6]:
# ============================================
# CELL 5: TECHNICAL INDICATORS
# ============================================

# --- MOVING AVERAGES ---
df['SMA_20']  = df['Close'].rolling(window=20).mean()  # 20-day SMA
df['SMA_50']  = df['Close'].rolling(window=50).mean()  # 50-day SMA
df['EMA_20']  = df['Close'].ewm(span=20, adjust=False).mean()  # Exponential MA

# --- BOLLINGER BANDS ---
bb_window = 20
df['BB_Mid']   = df['Close'].rolling(window=bb_window).mean()
bb_std         = df['Close'].rolling(window=bb_window).std()
df['BB_Upper']  = df['BB_Mid'] + (bb_std * 2)
df['BB_Lower']  = df['BB_Mid'] - (bb_std * 2)

# --- RSI (Relative Strength Index) ---
def calculate_rsi(series, period=14):
    delta  = series.diff()
    gain   = delta.clip(lower=0)
    loss   = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs     = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['RSI'] = calculate_rsi(df['Close'])

# --- MACD ---
ema12          = df['Close'].ewm(span=12, adjust=False).mean()
ema26          = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']      = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist']  = df['MACD'] - df['MACD_Signal']

# --- VOLUME MOVING AVERAGE ---
df['Volume_MA'] = df['Volume'].rolling(window=20).mean()

print("✅ All technical indicators calculated!")
df[['Date', 'Close', 'SMA_20', 'RSI', 'MACD']].tail(5)

✅ All technical indicators calculated!


,Date,Close,SMA_20,RSI,MACD
243,2026-04-15,3256.500000,3085.910034,59.585168,-20.798422
244,2026-04-16,3222.300049,3100.470032,64.529486,-13.476049
245,2026-04-17,3200.199951,3108.675024,59.777687,-9.348544
246,2026-04-20,3221.600098,3113.310034,55.930477,-4.301074
247,2026-04-21,3247.300049,3114.945032,64.163903,1.752651


In [7]:
fig = go.Figure(data=[
    go.Candlestick(
        x=df['Date'],
        open=df['Open'], high=df['High'],
        low=df['Low'],   close=df['Close'],
        name='M&M Price'
    ),
    go.Scatter(x=df['Date'], y=df['SMA_20'],
               name='SMA 20', line=dict(color='orange')),
    go.Scatter(x=df['Date'], y=df['SMA_50'],
               name='SMA 50', line=dict(color='cyan')),
])

fig.update_layout(
    title='Mahindra & Mahindra — 1 Year Candlestick Chart',
    xaxis_title='Date', yaxis_title='Price (₹)',
    template='plotly_dark', height=500,
    xaxis_rangeslider_visible=False
)
fig.show()

# --- RSI CHART ---
fig2 = px.line(df, x='Date', y='RSI',
               title='RSI (14-Day) — Overbought/Oversold',
               template='plotly_dark')
fig2.add_hline(y=70, line_dash="dash", line_color="red",
               annotation_text="Overbought (70)")
fig2.add_hline(y=30, line_dash="dash", line_color="green",
               annotation_text="Oversold (30)")
fig2.show()

# --- VOLUME BAR CHART ---
fig3 = px.bar(df, x='Date', y='Volume', color='Up_Down',
              color_discrete_map={'Up':'#2ecc71','Down':'#e74c3c'},
              title='Daily Volume (Green=Up Day, Red=Down Day)',
              template='plotly_dark')
fig3.show()

In [12]:
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

output_file = "/content/MM_Stock_Analysis_1Year.csv"
df.to_csv(output_file, index=False)

print(f"✅ CSV saved: {output_file}")
print(f"📦 Shape   : {df.shape}")
print(f"📋 Columns : {list(df.columns)}")

# Download to your computer
from google.colab import files
files.download(output_file)

print("⬇️  Download started! Check your Downloads folder.")

✅ CSV saved: /content/MM_Stock_Analysis_1Year.csv
📦 Shape   : (248, 28)
📋 Columns : ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Year', 'Month', 'Quarter', 'DayOfWeek', 'Week', 'Daily_Return', 'Cumulative_Return', 'Price_Change', 'Up_Down', 'SMA_20', 'SMA_50', 'EMA_20', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'Volume_MA']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download started! Check your Downloads folder.


In [10]:
# FIX DATE FORMAT FOR LOOKER STUDIO
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

# Save again
output_file = "/content/MM_Stock_Fixed.csv"
df.to_csv(output_file, index=False)

from google.colab import files
files.download(output_file)
print("✅ Fixed CSV downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Fixed CSV downloaded!
